# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [3]:
import os

path = r'C:\Users\juani\OneDrive\Escritorio\01_corpus_turismo_500.txt'

# Leer el archivo
with open(path, 'r', encoding='utf-8') as f:
    files = f.readlines()

In [4]:
# Eliminar líneas vacías y espacios al inicio/final
documentos_turismo = [line.strip() for line in files if line.strip()]

print(f"Número de documentos en el corpus de turismo: {len(documentos_turismo)}")

Número de documentos en el corpus de turismo: 500


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Creamos la matriz TF-IDF
matriz_turismo = TfidfVectorizer()

# Calcular la matriz TF-IDF
tfidf_matrix_turismo = matriz_turismo.fit_transform(documentos_turismo)

# Mostrar dimensiones de la matriz (documentos x términos)
print(f"Dimensión de la matriz TF-IDF: {tfidf_matrix_turismo.shape}")
print("Ejemplo de términos en el vocabulario:", list(matriz_turismo.vocabulary_.keys())[:10])

Dimensión de la matriz TF-IDF: (500, 116)
Ejemplo de términos en el vocabulario: ['otavalo', 'es', 'conocido', 'por', 'su', 'mercado', 'indígena', 'artesanía', 'perfecto', 'para']


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [12]:
import os

gutenberg_dir = "C:/Users/juani/OneDrive/Escritorio/gutenberg_1000"

def cargar_corpus_gutenberg(ruta_carpeta):
    documentos = []
    nombres = []

    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".txt"):
            ruta_completa = os.path.join(ruta_carpeta, archivo)

            with open(ruta_completa, "r", encoding="utf-8") as f:
                contenido = f.read()

            documentos.append(contenido)
            nombres.append(archivo)

    return documentos, nombres


documentos_gutenberg, nombres_gutenberg = cargar_corpus_gutenberg(gutenberg_dir)
print(f"Se cargaron {len(documentos_gutenberg)} documentos.")

Se cargaron 1000 documentos.


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Crear vectorizador TF-IDF
vectorizer_gutenberg = TfidfVectorizer(
    stop_words='english',   # elimina palabras comunes
    lowercase=True,
    max_features=5000       # limitar vocabulario (opcional)
)

# Calcular matriz TF-IDF
tfidf_matrix_gutenberg = vectorizer_gutenberg.fit_transform(documentos_gutenberg)

# Mostrar dimensiones
print("Dimensión de la matriz TF-IDF:", tfidf_matrix_gutenberg.shape)
print("Tamaño del vocabulario:", len(vectorizer_gutenberg.vocabulary_))


# Convertir a DataFrame para visualizar
df_tfidf_gutenberg = pd.DataFrame(
    tfidf_matrix_gutenberg.toarray(),
    columns=vectorizer_gutenberg.get_feature_names_out(),
    index=nombres_gutenberg
)

# Mostrar primeras filas
display(df_tfidf_gutenberg.head())

Dimensión de la matriz TF-IDF: (1000, 5000)
Tamaño del vocabulario: 5000


,00,000,047,10,100,101,102,103,104,105,...,él,én,és,étaient,était,état,étoit,été,être,über
book_10085.txt,0.0,0.002994,0.0,0.006807,0.002533,0.000000,0.000000,0.001126,0.000000,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.000000,0.00000,0.0
book_10142.txt,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.000000,0.00000,0.0
book_10356.txt,0.0,0.065203,0.0,0.020072,0.005746,0.002472,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.000000,0.00000,0.0
book_10357.txt,0.0,0.002121,0.0,0.009108,0.002659,0.001716,0.001773,0.002659,0.004456,0.002938,...,0.0,0.0,0.0,0.000000,0.001376,0.00000,0.0,0.000000,0.00000,0.0
book_10385.txt,0.0,0.000087,0.0,0.006393,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.061385,0.142208,0.01574,0.0,0.035344,0.04519,0.0


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [14]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def Buscar(consulta, top_n=10):
    """
    Busca documentos relevantes en Gutenberg 1000 usando TF-IDF + Cosine Similarity

    Parámetros:
        consulta (str): texto a buscar
        top_n (int): número de resultados

    Retorna:
        DataFrame con ranking
    """

    # convertir consulta a vector TF-IDF
    query_vector = vectorizer_gutenberg.transform([consulta])

    # calcular similitud coseno
    similitudes = cosine_similarity(
        query_vector,
        tfidf_matrix_gutenberg
    ).flatten()

    # construir ranking
    resultados = pd.DataFrame({
        "Documento": nombres_gutenberg,
        "Score": similitudes
    })

    # ordenar
    resultados = resultados.sort_values(
        by="Score",
        ascending=False
    ).reset_index(drop=True)

    resultados.index += 1

    return resultados.head(top_n)

In [15]:
# Prueba de la función con una consulta de ejemplo
Buscar("love romance marriage")

,Documento,Score
1,book_3692.txt,0.198344
2,book_12223.txt,0.179511
3,book_59057.txt,0.173151
4,book_67152.txt,0.154373
5,book_3665.txt,0.136075
6,book_2414.txt,0.128395
7,book_26368.txt,0.127044
8,book_45512.txt,0.118632
9,book_22343.txt,0.115253
10,book_35907.txt,0.113302
